<!--nav--> [🗺 Learning path](README.md) · **6/44** · ◀ [Modern Full FineTuning PostTraining](./Modern_Full_FineTuning_PostTraining.ipynb) · [Free Distributed SFT DPO](./Free_Distributed_SFT_DPO.ipynb) ▶

# Distributed DPO + LoRA with DeepSpeed & Ray

## What This Notebook Does

**One clean training run** — DPO alignment with LoRA, accelerated by DeepSpeed.

### The Stack

| Layer | Tool | What It Does |
|-------|------|--------------|
| **Training** | TRL DPOTrainer | DPO loss on preference pairs |
| **Efficiency** | LoRA (PEFT) | Train only ~1% of params |
| **Memory** | DeepSpeed ZeRO-2 | Shard optimizer + gradients to CPU |
| **Scale** | Ray Train | Orchestrate across GPUs/nodes |

### DeepSpeed ZeRO in 30 Seconds

Training a model needs: **model weights** + **gradients** + **optimizer states** (2x-8x model size).

```
ZeRO-0: Everything on every GPU              (baseline)
ZeRO-1: Shard optimizer states               (4x less memory)
ZeRO-2: Shard optimizer + gradients           (8x less memory)
ZeRO-3: Shard optimizer + gradients + model   (Nx less, N=GPUs)
```

We use **ZeRO-2 + CPU offload** — sweet spot of speed and memory savings.

### Ray in 30 Seconds

Ray distributes Python across machines. `Ray Train` wraps HuggingFace Trainer
so the same code scales from 1 GPU to 100 GPUs without code changes.

```
1 GPU Colab  →  Ray runs training locally (no overhead)
4 GPU server →  Ray launches 4 workers, DeepSpeed shards across them
Multi-node   →  Ray handles networking, DeepSpeed handles training
```

---
**Runtime:** Colab Pro (any GPU) or your own server

## Step 1: Install

In [ ]:
!pip install -q transformers datasets peft accelerate trl deepspeed
!pip install -q "ray[train]>=2.9"

## Step 2: Check GPU

In [ ]:
import torch
import os
import json
import gc
import time

os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "GPU required! Enable GPU in Runtime > Change runtime type."

NUM_GPUS = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPUs: {NUM_GPUS} x {GPU_NAME} ({GPU_MEM:.0f} GB)")

# Use TinyLlama — small enough for any GPU, fast to train
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Model: {MODEL_NAME}")

## Step 3: DeepSpeed Config

Just a JSON file. The `"auto"` values get filled in by HuggingFace Trainer.

In [ ]:
ds_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True,
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
}

with open("ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

print("DeepSpeed ZeRO-2 config saved.")
print("  - Optimizer states: offloaded to CPU")
print("  - Gradients: sharded across GPUs")
print("  - Model: stays on GPU (fast forward/backward)")

## Step 4: Preference Data for DPO

DPO needs pairs: a **chosen** (good) and **rejected** (bad) response for each prompt.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Preference pairs — chosen is better than rejected
PAIRS = [
    {
        "prompt": "Explain what a black hole is.",
        "chosen": "A black hole is a region in space where gravity is so strong that nothing, not even light, can escape. They form when massive stars collapse at the end of their life cycle.",
        "rejected": "A black hole is a hole that is black. It sucks things in. Nobody really knows what they are.",
    },
    {
        "prompt": "How do I make scrambled eggs?",
        "chosen": "Crack 2-3 eggs into a bowl, whisk with salt and pepper. Heat butter in a pan over medium-low heat, pour in eggs, and gently stir with a spatula until softly set.",
        "rejected": "Put eggs in pan. Cook them. Add stuff if you want.",
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed. For example, a spam filter learns from labeled emails.",
        "rejected": "Machine learning is when computers learn stuff. It's really complicated and uses lots of math.",
    },
    {
        "prompt": "Why is exercise important?",
        "chosen": "Regular exercise strengthens your heart, improves mood by releasing endorphins, helps maintain a healthy weight, and reduces the risk of chronic diseases like diabetes.",
        "rejected": "Exercise is good for you. You should do it because everyone says so.",
    },
    {
        "prompt": "Explain recursion in programming.",
        "chosen": "Recursion is when a function calls itself to solve smaller sub-problems. For example, factorial(5) = 5 * factorial(4). Every recursive function needs a base case to stop.",
        "rejected": "Recursion is a hard concept. It's when things repeat. You'll understand it eventually.",
    },
    {
        "prompt": "What is photosynthesis?",
        "chosen": "Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen. It happens in chloroplasts using chlorophyll, the pigment that makes plants green.",
        "rejected": "Plants eat sunlight somehow. It's a biology thing.",
    },
    {
        "prompt": "How does the internet work?",
        "chosen": "The internet is a global network of computers connected via cables and wireless links. When you visit a website, your browser sends a request to a server, which sends back the webpage data using protocols like HTTP/TCP.",
        "rejected": "The internet is wifi. You connect and it works. Nobody really knows the details.",
    },
    {
        "prompt": "What is the theory of relativity?",
        "chosen": "Einstein's theory of relativity has two parts: special relativity says the speed of light is constant and time slows at high speeds. General relativity says massive objects curve spacetime, which we experience as gravity.",
        "rejected": "Relativity is Einstein's thing about E=mc2. It means everything is relative.",
    },
]


def format_chat(prompt, response):
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]


rows = []
for ex in PAIRS:
    rows.append({
        "prompt": [{"role": "user", "content": ex["prompt"]}],
        "chosen": format_chat(ex["prompt"], ex["chosen"]),
        "rejected": format_chat(ex["prompt"], ex["rejected"]),
    })

dataset = Dataset.from_list(rows)
print(f"Dataset: {len(dataset)} preference pairs")

## Step 5: DPO + LoRA + DeepSpeed Training

This is the whole thing. Three additions to normal DPO training:
1. `LoraConfig` — train only ~1% of parameters
2. `deepspeed="ds_config.json"` — one line to enable DeepSpeed
3. That's it. There is no step 3.

In [ ]:
from transformers import AutoModelForCausalLM, TrainerCallback
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig


class LossTracker(TrainerCallback):
    """Collects loss values for plotting."""
    def __init__(self):
        self.losses = []
        self.start = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append({
                "step": state.global_step,
                "loss": logs["loss"],
                "sec": round(time.time() - self.start, 1),
            })


# Load model
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# DPO config — note the deepspeed line
tracker = LossTracker()

training_args = DPOConfig(
    output_dir="./dpo_deepspeed_output",
    beta=0.1,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    logging_steps=1,
    bf16=True,
    gradient_checkpointing=True,
    do_eval=False,
    remove_unused_columns=False,
    report_to="none",
    save_strategy="no",
    deepspeed="ds_config.json",  # <-- THE ONLY DEEPSPEED LINE
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    callbacks=[tracker],
)

# Train
print("\nTraining DPO + LoRA + DeepSpeed ZeRO-2...")
start = time.time()
trainer.train()
train_time = time.time() - start

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"\nDone! Time: {train_time:.1f}s | Peak GPU: {peak_mem:.1f} GB")

## Step 6: Test the Aligned Model

In [ ]:
test_prompts = [
    "What is gravity?",
    "How do I learn Python?",
    "What makes a good friend?",
]

model.eval()
responses = []

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    responses.append({"prompt": prompt, "response": response})
    print(f"\nQ: {prompt}")
    print(f"A: {response[:300]}")

## Step 7: Training Summary

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
if tracker.losses:
    steps = [l["step"] for l in tracker.losses]
    losses = [l["loss"] for l in tracker.losses]
    axes[0].plot(steps, losses, color="#34d399", linewidth=2, marker="o", markersize=3)
    axes[0].set_title("DPO Training Loss", fontsize=14)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.2)

# Memory bar
bars = axes[1].bar(
    ["Peak GPU\nUsed", "GPU\nTotal"],
    [peak_mem, GPU_MEM],
    color=["#34d399", "#4b5563"],
    edgecolor="white",
    linewidth=0.5,
)
axes[1].set_title("GPU Memory (GB)", fontsize=14)
axes[1].set_ylabel("GB")
for bar in bars:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f"{bar.get_height():.1f}", ha="center", fontsize=12, fontweight="bold",
    )

plt.suptitle(f"DPO + LoRA + DeepSpeed ZeRO-2 | {GPU_NAME}", fontsize=15, color="#a78bfa")
plt.tight_layout()
plt.show()

print(f"\nTraining time: {train_time:.1f}s")
print(f"Peak GPU memory: {peak_mem:.1f} / {GPU_MEM:.0f} GB")
print(f"Dataset: {len(dataset)} preference pairs")
print(f"DeepSpeed: ZeRO-2 + CPU optimizer offload")

In [ ]:
# Save model
trainer.save_model("./dpo_deepspeed_output/final")
tokenizer.save_pretrained("./dpo_deepspeed_output/final")
print("Model saved to ./dpo_deepspeed_output/final")

# Clean up DeepSpeed state before Ray section
del model, trainer
gc.collect()
torch.cuda.empty_cache()

---

# Part 2: Ray Train — Scale to Multiple GPUs

Ray Train wraps the exact same training code so it works on any number of GPUs.

**On Colab (1 GPU):** Ray runs it locally — same result, small overhead.  
**On your server (4+ GPUs):** Ray launches workers, DeepSpeed shards across them.

### How Ray + DeepSpeed Work Together

```
Ray Train                          DeepSpeed
--------                          ---------
Launches N workers (1 per GPU)    Each worker runs ZeRO
Handles networking/discovery      Handles gradient sharding
Manages checkpoints              Manages optimizer offload
Scales across machines           Scales within each machine
```

They're complementary: Ray handles **orchestration**, DeepSpeed handles **memory optimization**.

In [ ]:
import ray
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig


# DeepSpeed config as a dict — Ray workers can't access files from the main process
DS_CONFIG_DICT = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True,
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
}


def train_dpo_with_ray():
    """This function runs on each Ray worker (1 worker per GPU)."""
    import torch, os, time
    os.environ["WANDB_DISABLED"] = "true"

    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import DPOConfig, DPOTrainer
    from peft import LoraConfig
    from ray.train.huggingface.transformers import (
        RayTrainReportCallback,
        prepare_trainer,
    )

    MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Load tokenizer + model
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
    )

    # Same preference data
    pairs = [
        {"prompt": "Explain what a black hole is.",
         "chosen": "A black hole is a region in space where gravity is so strong that nothing, not even light, can escape.",
         "rejected": "A black hole is a hole that is black."},
        {"prompt": "How do I make scrambled eggs?",
         "chosen": "Crack 2-3 eggs into a bowl, whisk with salt and pepper. Heat butter in a pan over medium-low heat.",
         "rejected": "Put eggs in pan. Cook them."},
        {"prompt": "What is machine learning?",
         "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed.",
         "rejected": "Machine learning is when computers learn stuff."},
        {"prompt": "Why is exercise important?",
         "chosen": "Regular exercise strengthens your heart, improves mood by releasing endorphins, and reduces chronic disease risk.",
         "rejected": "Exercise is good for you. Everyone says so."},
        {"prompt": "Explain recursion in programming.",
         "chosen": "Recursion is when a function calls itself to solve smaller sub-problems. Every recursive function needs a base case.",
         "rejected": "Recursion is a hard concept. It's when things repeat."},
    ]

    def fmt(p, r):
        return [{"role": "user", "content": p}, {"role": "assistant", "content": r}]

    rows = [{"prompt": [{"role": "user", "content": e["prompt"]}],
             "chosen": fmt(e["prompt"], e["chosen"]),
             "rejected": fmt(e["prompt"], e["rejected"])} for e in pairs]
    ds = Dataset.from_list(rows)

    # DPO + LoRA config
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    )

    training_args = DPOConfig(
        output_dir="./ray_dpo_output",
        beta=0.1,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        warmup_steps=5,
        logging_steps=1,
        bf16=True,
        gradient_checkpointing=True,
        do_eval=False,
        remove_unused_columns=False,
        report_to="none",
        save_strategy="no",
        deepspeed=DS_CONFIG_DICT,  # <-- DICT not file path (Ray workers can't see files)
    )

    trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=ds,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[RayTrainReportCallback()],
    )

    # prepare_trainer sets up distributed communication
    trainer = prepare_trainer(trainer)
    trainer.train()


# Initialize Ray
if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)
print(f"Ray initialized. Resources: {ray.cluster_resources()}")

In [ ]:
# Launch training via Ray
# On Colab: 1 worker, 1 GPU
# On your server: change num_workers to match your GPU count

ray_trainer = TorchTrainer(
    train_loop_per_worker=train_dpo_with_ray,
    scaling_config=ScalingConfig(
        num_workers=NUM_GPUS,     # 1 worker per GPU
        use_gpu=True,
    ),
    run_config=RunConfig(
        name="dpo_lora_deepspeed",
    ),
)

print(f"Launching Ray Train with {NUM_GPUS} worker(s)...")
result = ray_trainer.fit()
print(f"\nRay training complete!")
print(f"Result: {result}")

ray.shutdown()

---

## Scaling to Your Own Server

The code above runs on Colab. To scale to your own multi-GPU server:

### Option 1: DeepSpeed launcher (simplest)
```bash
# 1 GPU
python train_dpo.py

# 4 GPUs
deepspeed --num_gpus=4 train_dpo.py

# 8 GPUs across 2 machines
deepspeed --num_gpus=4 --num_nodes=2 \
    --master_addr=192.168.1.1 train_dpo.py
```

### Option 2: Ray (for clusters)
```bash
# Start Ray head node
ray start --head

# On other machines, join the cluster
ray start --address='HEAD_IP:6379'

# Run training — Ray finds all GPUs automatically
python train_with_ray.py
```

### Option 3: Accelerate (HuggingFace native)
```bash
accelerate launch --num_processes=4 train_dpo.py
```

### Memory Scaling (7B model + LoRA)

| GPUs | ZeRO-2 Memory/GPU | Speed |
|------|-------------------|-------|
| 1 | ~18 GB | 1x |
| 2 | ~12 GB | ~1.8x |
| 4 | ~8 GB | ~3.5x |
| 8 | ~6 GB | ~7x |

In [ ]:
# Generate standalone script for multi-GPU use
script = '''#!/usr/bin/env python3
"""
DPO + LoRA + DeepSpeed — Multi-GPU Training Script

Usage:
  1 GPU:   python train_dpo_distributed.py
  4 GPUs:  deepspeed --num_gpus=4 train_dpo_distributed.py
"""
import torch, os, json
os.environ["WANDB_DISABLED"] = "true"

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# For bigger models, change to e.g. "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
)

# Your preference data here
pairs = [
    {"prompt": "Explain what a black hole is.",
     "chosen": "A black hole is a region in space where gravity is so strong that nothing can escape.",
     "rejected": "A black hole is a hole that is black."},
]

def fmt(p, r):
    return [{"role": "user", "content": p}, {"role": "assistant", "content": r}]

rows = [{"prompt": [{"role": "user", "content": e["prompt"]}],
         "chosen": fmt(e["prompt"], e["chosen"]),
         "rejected": fmt(e["prompt"], e["rejected"])} for e in pairs]
dataset = Dataset.from_list(rows)

trainer = DPOTrainer(
    model=model,
    args=DPOConfig(
        output_dir="./dpo_output",
        beta=0.1,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        bf16=True,
        gradient_checkpointing=True,
        do_eval=False,
        remove_unused_columns=False,
        report_to="none",
        deepspeed="ds_config.json",
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    ),
)

trainer.train()
trainer.save_model("./dpo_output/final")
print("Done!")
'''

with open("train_dpo_distributed.py", "w") as f:
    f.write(script)

print("Saved: train_dpo_distributed.py")
print("")
print("To run on your server:")
print("  1 GPU:   python train_dpo_distributed.py")
print("  4 GPUs:  deepspeed --num_gpus=4 train_dpo_distributed.py")

---

## Cheat Sheet

### What We Used

| Concept | What | Why |
|---------|------|-----|
| **DPO** | Train on preference pairs (chosen vs rejected) | Align model without reward model or RL |
| **LoRA** | Train tiny adapter matrices (~1% params) | Fits on small GPUs, fast training |
| **DeepSpeed ZeRO-2** | Shard optimizer + gradients to CPU | 8x memory reduction |
| **Ray Train** | Distribute training across GPUs | Same code scales from 1 to N GPUs |

### When to Use What

| Situation | Use |
|-----------|-----|
| 1 GPU, model fits | Standard training (no DeepSpeed) |
| 1 GPU, tight on memory | DeepSpeed ZeRO-2 + CPU offload |
| 1 GPU, model doesn't fit | DeepSpeed ZeRO-3 + CPU offload |
| Multiple GPUs, same machine | `deepspeed --num_gpus=N` |
| Multiple machines | Ray Train + DeepSpeed |

### The Key Insight

Adding DeepSpeed to any HuggingFace training is **one line**:
```python
DPOConfig(..., deepspeed="ds_config.json")
```
Everything else stays the same.